# Agente Funcional con Memoria y Herramientas de Escritura

Este notebook implementa un agente funcional que integra:
- **Herramientas de consulta**: busqueda semantica en lore, validacion de personajes, verificacion temporal.
- **Herramienta de escritura**: genera y guarda reportes de continuidad en formato Markdown.
- **Memoria conversacional**: mantiene historial de interacciones para responder preguntas de seguimiento.

Framework: LangChain `create_openai_tools_agent` + `ConversationBufferMemory`.

In [ ]:
import os
import re
import json
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain import hub
from langchain.memory import ConversationBufferMemory
from langchain_core.messages import AIMessage, HumanMessage

from lore_database import get_all_fragments, get_fragments_by_category, LoreFragment

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

## Motor de busqueda semantica

Reutilizado del agente original (EP1). Busca por overlap de palabras clave y tags.

In [ ]:
def _buscar(query: str, fragments: list[LoreFragment], top_k=3) -> list[str]:
    query_words = set(re.sub(r"[^\w\s]", "", query.lower()).split())
    scored = []
    for frag in fragments:
        text_words = set(re.sub(r"[^\w\s]", "", frag.text.lower()).split())
        tag_words = set(frag.tags)
        overlap = len(query_words & (text_words | tag_words))
        if overlap > 0:
            scored.append((overlap, frag.text))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [t for _, t in scored[:top_k]]

## Herramientas de consulta

Tres herramientas para recuperar informacion del lore canonico.

In [ ]:
@tool
def search_lore(query: str) -> str:
    """Busca informacion en el lore canonico de Transformers. Usa para verificar hechos generales."""
    results = _buscar(query, get_all_fragments(), top_k=3)
    if not results:
        return "No se encontraron resultados para esa consulta."
    out = f"Lore recuperado para '{query}':\n\n"
    for i, t in enumerate(results, 1):
        out += f"[{i}] {t}\n\n"
    return out.strip()

@tool
def validate_character(character_name: str) -> str:
    """Verifica el estado canonico de un personaje: apariciones, habilidades, muerte, afiliacion."""
    frags = get_fragments_by_category("PERSONAJE")
    name_lower = character_name.lower()
    relevant = [f.text for f in frags if name_lower in f.text.lower() or any(name_lower in tag for tag in f.tags)]
    if not relevant:
        relevant = _buscar(character_name, get_all_fragments(), top_k=2)
    if not relevant:
        return f"Personaje '{character_name}' no encontrado en el lore."
    return f"Estado canonico de '{character_name}':\n\n" + "\n\n".join(f"- {t}" for t in relevant)

@tool
def check_timeline(year_and_event: str) -> str:
    """Valida si un evento es coherente con la linea temporal de la saga. Formato: '2009 - descripcion del evento'."""
    frags = get_fragments_by_category("TIMELINE") + get_fragments_by_category("EVENTO")
    results = _buscar(year_and_event, frags, top_k=4)
    if not results:
        return "No se encontraron eventos canonicos para ese periodo."
    return f"Timeline canonico para '{year_and_event}':\n\n" + "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(results))

## Herramienta de escritura

Genera reportes de continuidad en Markdown y los persiste en `/reports`.

In [ ]:
@tool
def write_continuity_report(report_data: str) -> str:
    """Genera y guarda un reporte de continuidad en formato Markdown. Entrada: JSON con fragment, verdict, inconsistencies, recommendation, confidence."""
    try:
        data = json.loads(report_data)
    except json.JSONDecodeError:
        return "Error: El reporte no esta en formato JSON valido."

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"continuity_report_{timestamp}.md"
    filepath = REPORTS_DIR / filename

    md_content = f"""# Reporte de Continuidad Narrativa — Transformers

**Fecha:** {datetime.now().isoformat()}  \n**ID:** {timestamp}

---

## Fragmento Analizado

> {data.get('fragment', 'No especificado')}

## Veredicto

**{data.get('verdict', 'PENDIENTE')}** — Confianza: {data.get('confidence', 0.0):.0%}

## Inconsistencias Detectadas

"""
    inconsistencies = data.get("inconsistencias") or data.get("inconsistencies", [])
    if inconsistencies:
        for idx, inc in enumerate(inconsistencies, 1):
            md_content += f"""### {idx}. {inc.get('description', 'Sin descripcion')}
- **Severidad:** {inc.get('severity', 'NO ESPECIFICADA')}
- **Guion dice:** {inc.get('guion_says', inc.get('guion_says', 'N/A'))}
- **Lore dice:** {inc.get('lore_says', 'N/A')}
- **Fuente lore:** {inc.get('lore_source', 'N/A')}

"""
    else:
        md_content += "_No se detectaron inconsistencias._\n\n"

    md_content += f"""## Recomendacion

{data.get('recommendation', 'Sin recomendacion.')}

---

*Generado por el Consultor de Continuidad Narrativa v2*
"""

    filepath.write_text(md_content, encoding="utf-8")
    return f"Reporte guardado en: {filepath.name}"

## Agente con memoria

Usa `ConversationBufferMemory` para mantener historial de conversacion.
Prompt oficial de LangChain para agentes con tools (`hwchase17/openai-tools-agent`).

In [ ]:
class ContinuityAgentV2:

    def __init__(self, verbose=True):
        self.llm = ChatOpenAI(
            base_url=os.getenv("OPENAI_BASE_URL"),
            api_key=os.getenv("GITHUB_TOKEN"),
            model="gpt-4o",
            temperature=0.1,
        )
        self.tools = [search_lore, validate_character, check_timeline, write_continuity_report]
        self.prompt = hub.pull("hwchase17/openai-tools-agent")
        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="output",
        )
        self.agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        self.executor = AgentExecutor(
            agent=self.agent,
            tools=self.tools,
            verbose=verbose,
            max_iterations=8,
            handle_parsing_errors=True,
            return_intermediate_steps=True,
            memory=self.memory,
        )

    def run(self, fragment: str):
        print(f"\n{'='*60}\nAnalizando: \"{fragment[:70]}...\"\n{'='*60}")
        result = self.executor.invoke({"input": fragment})
        print(f"\nVeredicto final:\n{result['output']}\n{'='*60}\n")
        return result

    def chat(self, message: str):
        print(f"\n[Usuario]: {message}")
        result = self.executor.invoke({"input": message})
        print(f"[Agente]: {result['output']}\n")
        return result

## Ejecucion de demos

Ejemplo 1: Fragmento inconsistente (Bumblebee habla en 2007).
Ejemplo 2: Pregunta de seguimiento sobre el analisis anterior (demuestra memoria).

In [ ]:
agent = ContinuityAgentV2(verbose=True)

print("\n" + "="*60)
print("DEMO: Agente con Memoria y Herramientas de Escritura")
print("="*60)

agent.run(
    "Escena 3 - 2007: Bumblebee se dirige a Sam y dice claramente: '
    'Sam, debes proteger el AllSpark. Confia en mi.'"
)

In [ ]:
agent.chat("Cual fue la inconsistencia principal que detectaste en el ultimo guion?")

In [ ]:
agent.run(
    "2009. Sam Witwicky y Cade Yeager trabajan juntos en el cuartel de NEST '
    'para analizar los fragmentos del AllSpark."
)

In [ ]:
agent.chat("En que pelicula aparece Cade Yeager y por que no podria estar en 2009?")